In [86]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = [file.parse() for file in reader.read()]

In [87]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

from pydantic import BaseModel
class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

prompt = """
Question:
{question}

File Name:
{filename}

Content:
{content}
""".strip()

import json
from evaluation_utils import llm_structured

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
        
    out, usage = llm_structured(
        openrouter_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="poolside/laguna-m.1:free"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

def average_input_tokens(n):
    sum = 0
    for i in range(0,n):
        sum += generate_ground_truth(documents[i])[1].input_tokens
    return sum/n

average_input_tokens(3)

1345.0

In [88]:
import pandas as pd
df_ground_truth = pd.read_csv("data/homework/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [89]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

from minsearch import Index
index_chunks = Index(text_fields=["content"],)
index_chunks.fit(chunks)
def text_search(query, num_results=5):
    return index_chunks.search(query, num_results=num_results)


from minsearch import VectorSearch
import numpy as np
from embedder import Embedder
embed = Embedder()
texts = [chunk["content"] for chunk in chunks]
X = []
batch_vectors = embed.encode_batch(texts)
X.extend(batch_vectors)
X = np.array(X)
vindex = VectorSearch()
vindex.fit(X, chunks)

def vector_search(query, num_results=5):    
    v_query = embed.encode(query)
    return vindex.search(v_query, num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=10)

In [90]:
q = ground_truth[0]["question"]
text_search(q)[0]["filename"], vector_search(q)[0]["filename"]

('01-agentic-rag/lessons/03-rag.md', '01-agentic-rag/lessons/01-intro.md')

In [91]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

from tqdm.auto import tqdm
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

evaluate(ground_truth, text_search), evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

({'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594},
 {'hit_rate': 0.725, 'mrr': 0.5486111111111112})

In [92]:
def hybrid_search_1(query):
    return hybrid_search(query, k=1)
    
def hybrid_search_50(query):
    return hybrid_search(query, k=50)
    
def hybrid_search_100(query):
    return hybrid_search(query, k=100)
    
def hybrid_search_200(query):
    return hybrid_search(query, k=200)

evaluate(ground_truth, hybrid_search_1), evaluate(ground_truth, hybrid_search_50), evaluate(ground_truth, hybrid_search_100), evaluate(ground_truth, hybrid_search_200)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

({'hit_rate': 0.9083333333333333, 'mrr': 0.6576917989417991},
 {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801},
 {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801},
 {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801})